# Notebook 07 — Power BI Export & Investor ROI

**Phase 7 — the final phase.** Everything we built in Phases 1-6 gets packaged into a single SQLite database that Power BI (or any BI tool) can connect to directly.

## What you will do here

1. Reload all assets from previous phases (model, data, SHAP values).
2. Build the **investor ROI table** — cap rate, cash-on-cash, and IRR for every home.
3. Assemble a **SQLite database** with five tables ready for Power BI.
4. Inspect and verify every table.
5. Learn how to connect Power BI to the database.

## Tables in the database

| Table | What it contains | Source phase |
|-------|------------------|--------------|
| `ames_predictions` | Predicted vs. actual price for every home | 4 |
| `shap_importance` | Feature importance ranking (mean \|SHAP\|) | 6 |
| `investment_roi` | Cap rate, cash-on-cash, IRR per home | 7 |
| `zhvi_forecasts` | Iowa ZIP 24-month price forecasts | 5 |

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import warnings
warnings.filterwarnings('ignore')

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)

from src.data_loader import load_ames, load_zillow_zhvi
from src.features import prepare_modeling_data
from src import models as M
from src import interpretability as I
from src import exporter as E

print('Setup complete.')

## 1. Reload all assets from previous phases

We reconstruct the same objects from Phases 4 and 6. Nothing is retrained — we load the saved model and recompute SHAP values on the test set.

In [ ]:
# Phase 4: model + data
model = M.load_model('xgb_ames_tuned')
X, y  = prepare_modeling_data(load_ames(), zhvi=load_zillow_zhvi())
X_train, X_test, y_train, y_test = M.split(X, y)
print(f'Model: {type(model.named_steps["model"]).__name__}')
print(f'All homes: {X.shape[0]:,}  |  Test set: {X_test.shape[0]:,}  |  Features: {X.shape[1]}')

# Phase 6: SHAP values on the test set
print('\nComputing SHAP values...')
explainer = I.get_explainer(model)
sv = I.shap_values(explainer, X_test)
print(f'SHAP values shape: {sv.shape}')

## 2. Investor ROI metrics

For each home, we estimate three investor metrics using standard real estate assumptions:

| Metric | Formula | What it tells you |
|--------|---------|-------------------|
| **Cap Rate** | NOI ÷ Property Value | All-cash yield before financing |
| **Cash-on-Cash** | Annual Cash Flow ÷ Cash Invested | Return on your actual down payment |
| **IRR** | Discount rate where NPV = 0 | Total return including appreciation & sale proceeds over 10 years |

The assumptions below are editable — change them to model different market conditions.

In [ ]:
pred_usd = np.expm1(model.predict(X))

roi_df = E.compute_investor_roi(
    pred_usd,
    gross_rent_multiplier=150,   # monthly rent = value / 150
    vacancy_rate=0.05,           # 5% vacancy
    expense_ratio=0.40,          # 40% of gross rent goes to expenses
    down_payment_pct=0.20,       # 20% down
    mortgage_rate=0.07,          # 7% annual mortgage rate
    loan_term_years=30,
    appreciation_rate=0.03,      # 3% annual appreciation assumed
    hold_years=10,
)

print(f'ROI table: {roi_df.shape}')
roi_df.head()

### ROI distribution

Visualize how cap rate and IRR are distributed across all homes. This tells you what's "normal" for the Ames market and which homes are outliers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label in zip(
    axes,
    ['cap_rate_pct', 'cash_on_cash_pct', 'irr_pct'],
    ['Cap Rate (%)', 'Cash-on-Cash (%)', 'IRR (%)'],
):
    data = roi_df[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='tomato', linestyle='--', linewidth=1.5, label=f'Median {data.median():.2f}%')
    ax.set_title(label)
    ax.set_xlabel('%')
    ax.set_ylabel('Homes')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Investor ROI Distribution — All Ames Homes', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Summary statistics:')
roi_df[['cap_rate_pct', 'cash_on_cash_pct', 'irr_pct']].describe().round(2)

### Top 10 homes by IRR

In [ ]:
top10 = (
    roi_df
    .sort_values('irr_pct', ascending=False)
    .head(10)[['home_id', 'pred_value_usd', 'monthly_rent', 'cap_rate_pct', 'cash_on_cash_pct', 'irr_pct']]
    .reset_index(drop=True)
)
top10.index += 1
top10

## 3. Build the SQLite database

`build_sqlite` assembles all four tables into `data/outputs/real_estate.db`. Power BI connects to this single file.

In [ ]:
db_path = E.build_sqlite(
    model, X, y,
    sv=sv,
    X_test=X_test,
    y_test=y_test,
)

## 4. Verify the database

In [ ]:
E.print_export_summary(db_path)

In [ ]:
# Spot-check: read one table back from SQLite
con = sqlite3.connect(db_path)
pd.read_sql('SELECT * FROM ames_predictions ORDER BY abs_error_usd LIMIT 10', con)

In [ ]:
# Top 10 features by SHAP importance from the database
pd.read_sql('SELECT * FROM shap_importance LIMIT 10', con)

In [ ]:
con.close()
print('Database connection closed.')

## 5. All exported files

Everything Power BI needs is now in `data/outputs/`.

In [ ]:
outputs_dir = PROJECT_ROOT / 'data' / 'outputs'
print('Files in data/outputs/:')
for f in sorted(outputs_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<35} {size_mb:.2f} MB')

## 6. Connecting Power BI to the database

### Option A — SQLite (recommended)

1. Install the free **SQLite ODBC Driver**: https://www.ch-werner.de/sqliteodbc/
2. In Power BI Desktop → **Get Data** → **ODBC**
3. Choose your SQLite DSN and point it at `data/outputs/real_estate.db`
4. All four tables appear automatically with their relationships.

### Option B — CSV files

If you prefer, each table was also saved as a CSV in `data/outputs/`:

| CSV | Table |
|-----|-------|
| `ames_predictions.csv` | Predicted vs. actual prices |
| `shap_importance.csv` | Feature importance ranking |
| `investment_roi.csv` | Cap rate / cash-on-cash / IRR |
| `iowa_zhvi_forecasts.csv` | ZIP-level price forecasts |

In Power BI → **Get Data** → **Text/CSV** → select each file.

### Suggested Power BI visuals

| Visual | Table | Fields |
|--------|-------|--------|
| Scatter: Actual vs. Predicted | `ames_predictions` | `actual_usd`, `pred_usd` |
| Bar: Top features | `shap_importance` | `feature`, `mean_abs_shap` |
| Map: ZIP forecast | `zhvi_forecasts` | `zip`, `forecast`, `date` |
| Table: Best investments | `investment_roi` | `pred_value_usd`, `cap_rate_pct`, `irr_pct` |
| Histogram: Error distribution | `ames_predictions` | `pct_error` |

## 7. Wrap-up — project complete!

You have now completed all 7 phases of the Real Estate Market Analyzer:

| Phase | What you built |
|-------|----------------|
| 1 | Data ingestion pipeline (Ames + Zillow) |
| 2 | Data cleaning and EDA |
| 3 | Feature engineering (aggregations, cyclic, cross-dataset) |
| 4 | ML models: Ridge, Random Forest, XGBoost + tuning |
| 5 | Time-series forecasting with SARIMA (per ZIP) |
| 6 | SHAP interpretability + renovation ROI simulator |
| 7 | Investor ROI metrics + SQLite database for Power BI |

### Final questions

1. **Cap rate vs. IRR.** A home has a cap rate of 5% but an IRR of 12%. How is that possible? What drives the gap between them?

2. **Sensitivity analysis.** What happens to IRR if the appreciation rate drops from 3% to 0%? Change `appreciation_rate=0.0` and re-run cell 5. Does the ranking of top homes stay the same?

3. **Data freshness.** The Ames data is from 2006-2010. In what ways does that limit the usefulness of the investor ROI table for a buyer today? What would you need to make it current?

4. **Power BI design.** If you were presenting this dashboard to a non-technical real estate investor, which two visuals would you put on the first page? Why?

5. **What's next?** Name one extension to this project that would make it genuinely useful for a real investor — something that would require new data, a new model, or a new metric not currently in the pipeline.